# 02 - Limpiar los viajes: tabla silver

Lee `viajes_bronze` (donde todo es texto) y arma `viajes_silver`, con los datos limpios. Hace cuatro cosas:

1. **Pone cada columna con su tipo correcto.** `fecha_hora` pasa a ser fecha y hora, `cantidad_pasajeros` y `ordinal_de_tramo` pasan a ser números enteros, `con_tarjeta` pasa a ser verdadero o falso, y a los textos se les sacan los espacios de más.
2. **Agrega columnas nuevas** a partir de la fecha: `fecha`, `hora`, `dia_semana` (1 es lunes y 7 es domingo) y `es_fin_de_semana`.
3. **Borra las filas que están mal:** fecha que no se puede leer, fecha de otro mes distinto al del archivo, pasajeros vacíos o menores a 1, o línea vacía.
4. **Borra las filas repetidas** (filas exactamente iguales a otra).

Cuántas filas se borran por cada motivo queda guardado por mes en la tabla `calidad_silver`. Así lo podemos contar en el informe.

Se procesa **un mes por vez**. Los meses que ya están en silver se saltean, salvo que pongas `FORZAR_SILVER = True`.

In [0]:
%run ./00_setup

# 00 - Preparar el proyecto

Este notebook deja todo listo para que los demás funcionen. Hace cuatro cosas:

1. Le dice a Python dónde está nuestro código (la carpeta `src/bondis`), para poder escribir `from bondis import ...`.
2. Crea en Databricks el lugar donde se guardan las tablas y los archivos (`bondis_stm.datos.raw`, definido en `config.py`).
3. Crea una carpeta para cada tipo de archivo (viajes, clima, etc.).
4. Muestra cómo dividimos los datos en train, val y test.

Se puede correr todas las veces que quieras. Si algo ya existe, no lo vuelve a crear ni lo borra.

No hace falta abrirlo a mano: los otros notebooks lo corren al principio con `%run ./00_setup`.

## 1. Encontrar nuestro código

Python solo puede importar código de las carpetas que conoce. Esta celda busca la carpeta del repositorio subiendo carpeta por carpeta hasta encontrar `src/bondis`, y la agrega a la lista de carpetas que Python conoce (`sys.path`).

## 2. Crear el catálogo, el esquema y el volume

Databricks ordena todo en tres niveles, como carpetas una dentro de otra:

- **Catálogo** (`bondis_stm`): la carpeta grande del proyecto.
- **Esquema** (`datos`): una subcarpeta donde van las tablas.
- **Volume** (`raw`): una carpeta para guardar archivos sueltos, como los ZIP y CSV que bajamos.


## 3. Crear una carpeta para cada tipo de archivo

| Carpeta | Qué va adentro |
|---|---|
| `viajes_zip/` | Los ZIP de viajes de cada mes, bajados a mano del portal de la Intendencia |
| `viajes/` | Los CSV de viajes, ya sacados de los ZIP |
| `clima/` | Lluvia y temperatura de cada hora |
| `calendario/` | Por ahora vacía. El calendario está escrito en `src/bondis/calendario.py` |
| `paros/` | Días y horas con paro (todavía no lo hicimos) |

## 4. Revisar que todo esté bien

Muestra la versión de Spark, el usuario y cuántos archivos hay en cada carpeta. Sirve para ver de un vistazo si falta subir algo.

## 5. Cómo dividimos los datos

Usamos las fechas para dividir, no el azar, porque queremos predecir el futuro con el pasado. Un año entero para cada cosa:

| Conjunto | Fechas | Para qué se usa |
|---|---|---|
| `train` | todo 2024 | El modelo aprende con estos datos |
| `val` | todo 2025 | Comparamos y elegimos los parámetros de la regresión |
| `test` | 2026 (lo que haya) | Probamos el modelo elegido una sola vez, al final |

La fecha de fin **no** está incluida. Por eso siempre se filtra con `fecha >= inicio` y `fecha < fin`.

`train_final` es train + val juntos. Cuando ya elegimos el modelo, lo volvemos a entrenar con 2024 y 2025 antes de probarlo en 2026.

## Opciones

In [0]:
MESES = None     # None quiere decir todos los meses que están en bronze
FORZAR_SILVER = False

In [0]:
import importlib

from pyspark.sql import functions as F

from bondis import silver

importlib.reload(silver)

silver.configurar_sesion(spark, config.TIMEZONE)
print("Zona horaria:", spark.conf.get("spark.sql.session.timeZone"))
print("Política de parseo de fechas:", spark.conf.get("spark.sql.legacy.timeParserPolicy"))


def mes_elegido(anio_mes):
    return MESES is None or anio_mes in MESES

Zona horaria: America/Montevideo
Política de parseo de fechas: CORRECTED


## 1. Prueba antes de empezar: ¿se pueden leer las fechas?

Toma 10.000 filas del primer mes y trata de convertir el texto de la fecha en una fecha de verdad.
Si menos del 99 % se puede convertir, el notebook se frena y muestra ejemplos de cómo vienen escritas las fechas.
En ese caso hay que agregar esa forma de escribir la fecha en `silver.FORMATOS_FECHA`.

Es mejor que falle acá, con 10.000 filas, que después de procesar millones.

In [0]:
bronze = spark.table(config.TABLE_BRONZE)
meses_bronze = sorted(r["anio_mes"] for r in bronze.select("anio_mes").distinct().collect())
meses_a_revisar = [am for am in meses_bronze if mes_elegido(am)]
assert meses_a_revisar, f"Ningún mes de MESES está en bronze. Meses disponibles: {meses_bronze}"

muestra = bronze.where(F.col("anio_mes") == meses_a_revisar[0]).limit(10000)
prueba = (
    silver.tipar(muestra)
    .agg(F.count("*").alias("filas"), F.count("fecha_hora").alias("fechas_ok"))
    .first()
)
tasa = prueba["fechas_ok"] / prueba["filas"]
print(f"Muestra de {meses_a_revisar[0]}: {prueba['fechas_ok']:,} de {prueba['filas']:,} fechas convertidas ({tasa:.1%})")

if tasa < 0.99:
    display(muestra.select("fecha_evento").limit(20))
    raise ValueError(
        "Menos del 99 % de las fechas se pudo convertir. Revisar el formato de 'fecha_evento' "
        "en la tabla de arriba y agregarlo a silver.FORMATOS_FECHA."
    )

Muestra de 2024-01: 10,000 de 10,000 fechas convertidas (100.0%)


## 2. ¿Qué meses faltan procesar?

In [0]:
ya_en_silver = set()
if spark.catalog.tableExists(config.TABLE_SILVER):
    ya_en_silver = {
        r["anio_mes"] for r in spark.table(config.TABLE_SILVER).select("anio_mes").distinct().collect()
    }

pendientes = [am for am in meses_a_revisar if FORZAR_SILVER or am not in ya_en_silver]
print(f"Meses en bronze:          {len(meses_bronze)}")
print(f"Meses ya en silver:       {len(ya_en_silver & set(meses_a_revisar))}")
print(f"Meses a procesar ahora:   {len(pendientes)}  {pendientes}")

Meses en bronze:          34
Meses ya en silver:       34
Meses a procesar ahora:   0  []


## 3. Limpiar mes por mes

Para cada mes:
1. Pone los tipos correctos (`tipar`).
2. Marca qué filas pasan cada control (`marcar_calidad`).
3. Cuenta cuántas filas fallan cada control.
4. Se queda con las filas buenas y saca las repetidas (`limpiar`).
5. Guarda el mes en `viajes_silver`, reemplazando solo ese mes.
6. Guarda los conteos del mes en `calidad_silver`.

In [0]:
def escribir_particion(df, tabla, anio_mes):
    """Guarda solo el mes anio_mes en la tabla, sin tocar los otros meses. Si la tabla no existe, la crea."""
    escritor = df.write.format("delta").mode("overwrite").option("mergeSchema", "true")
    if spark.catalog.tableExists(tabla):
        escritor.option("replaceWhere", f"anio_mes = '{anio_mes}'").saveAsTable(tabla)
    else:
        escritor.partitionBy("anio_mes").saveAsTable(tabla)


for am in pendientes:
    marcado = silver.marcar_calidad(silver.tipar(bronze.where(F.col("anio_mes") == am)))

    resumen = silver.resumen_calidad(marcado).first().asDict()
    limpio = silver.limpiar(marcado)
    escribir_particion(limpio, config.TABLE_SILVER, am)

    filas_silver = spark.table(config.TABLE_SILVER).where(F.col("anio_mes") == am).count()
    resumen.update(
        anio_mes=am,
        duplicados=resumen["filas_validas"] - filas_silver,
        filas_silver=filas_silver,
        pct_descartado=round(100 * (1 - filas_silver / resumen["filas_bronze"]), 3),
    )
    escribir_particion(spark.createDataFrame([resumen]), config.TABLE_CALIDAD_SILVER, am)

    print(f"[{am}] bronze {resumen['filas_bronze']:>12,}  ->  silver {filas_silver:>12,}  "
          f"({resumen['pct_descartado']:.3f} % descartado)")

## 4. Cuántas filas se borraron y por qué

Una fila por mes. Qué quiere decir cada columna:

| Columna | Por qué se borró la fila |
|---|---|
| `fecha_invalida` | La fecha no se pudo leer |
| `fecha_fuera_de_mes` | La fecha es de otro mes, no del mes del archivo |
| `pasajeros_invalidos` | `cantidad_pasajeros` está vacía, no es un número o es menor a 1 |
| `linea_nula` | La línea está vacía |
| `duplicados` | La fila era igual a otra |

Una misma fila puede fallar en más de un control. Por eso, si sumás los motivos, te puede dar más que el total de filas borradas.

In [0]:
if spark.catalog.tableExists(config.TABLE_CALIDAD_SILVER):
    display(
        spark.table(config.TABLE_CALIDAD_SILVER)
        .select("anio_mes", "filas_bronze", "fecha_invalida", "fecha_fuera_de_mes",
                "pasajeros_invalidos", "linea_nula", "duplicados", "filas_silver",
                "pct_descartado", "pasajeros_validos")
        .orderBy("anio_mes")
    )

anio_mes,filas_bronze,fecha_invalida,fecha_fuera_de_mes,pasajeros_invalidos,linea_nula,duplicados,filas_silver,pct_descartado,pasajeros_validos
2024-01,19054962,0,0,0,0,0,19054962,0.0,20405449
2024-02,19966497,0,0,0,0,0,19966497,0.0,21208320
2024-03,23408053,0,0,0,0,0,23408053,0.0,24353197
2024-04,26296998,0,0,0,0,0,26296998,0.0,27054556
2024-05,26281987,0,0,0,0,0,26281987,0.0,27012633
2024-06,25182931,0,0,0,0,0,25182931,0.0,25979315
2024-07,24547115,0,0,0,0,0,24547115,0.0,25423052
2024-08,26079619,0,0,0,0,0,26079619,0.0,26817782
2024-09,25615090,0,0,0,0,0,25615090,0.0,26501747
2024-10,28214243,0,0,0,0,0,28214243,0.0,29111351


## 5. Revisión de silver

Primero se muestran las columnas y sus tipos. Después, los pasajeros por hora en días de semana y fines de semana.

In [0]:
s = spark.table(config.TABLE_SILVER)
s.printSchema()

root
 |-- id_viaje: string (nullable = true)
 |-- tipo_viaje: string (nullable = true)
 |-- descripcion_tipo_viaje: string (nullable = true)
 |-- grupo_usuario: string (nullable = true)
 |-- descripcion_grupo_usuario: string (nullable = true)
 |-- grupo_usuario_especifico: string (nullable = true)
 |-- descripcion_grupo_usuario_espe: string (nullable = true)
 |-- codigo_parada_origen: string (nullable = true)
 |-- cod_empresa: string (nullable = true)
 |-- descrip_empresa: string (nullable = true)
 |-- linea_codigo: string (nullable = true)
 |-- dsc_linea: string (nullable = true)
 |-- sevar_codigo: string (nullable = true)
 |-- fecha_hora: timestamp (nullable = true)
 |-- cantidad_pasajeros: integer (nullable = true)
 |-- ordinal_de_tramo: integer (nullable = true)
 |-- con_tarjeta: boolean (nullable = true)
 |-- anio_mes: string (nullable = true)
 |-- archivo_origen: string (nullable = true)
 |-- fecha: date (nullable = true)
 |-- hora: integer (nullable = true)
 |-- dia_semana: inte

In [0]:
# Pasajeros por hora del primer mes procesado, separando días de semana y fines de semana.
# Si los datos están bien, en días de semana se tienen que ver dos picos: a la mañana y a la tarde.
mes_ejemplo = (pendientes or meses_a_revisar)[0]
display(
    s.where(F.col("anio_mes") == mes_ejemplo)
    .groupBy("es_fin_de_semana", "hora")
    .agg(F.sum("cantidad_pasajeros").alias("pasajeros"))
    .orderBy("es_fin_de_semana", "hora")
)

es_fin_de_semana,hora,pasajeros
false,0,95194
false,1,35842
false,2,15811
false,3,12488
false,4,78834
false,5,343592
false,6,633429
false,7,945987
false,8,1114141
false,9,993614


In [0]:
display(s.where(F.col("anio_mes") == mes_ejemplo).limit(10))

id_viaje,tipo_viaje,descripcion_tipo_viaje,grupo_usuario,descripcion_grupo_usuario,grupo_usuario_especifico,descripcion_grupo_usuario_espe,codigo_parada_origen,cod_empresa,descrip_empresa,linea_codigo,dsc_linea,sevar_codigo,fecha_hora,cantidad_pasajeros,ordinal_de_tramo,con_tarjeta,anio_mes,archivo_origen,fecha,hora,dia_semana,es_fin_de_semana
4265536,10,JUB. A,3,JUBILADO,31,JUBILADO A,3735,20,C.O.M.E.S.A,16,522,2426,2024-01-11T10:06:26.000-03:00,1,2,true,2024-01,viajes_stm_2024_01.csv,2024-01-11,10,4,false
13210185,5,1 HORA,1,USUARIO CORRIENTE,1,USUARIO CORRIENTE,4925,50,C.U.T.C.S.A.,116,L4,1579,2024-01-29T17:26:00.000-03:00,1,2,true,2024-01,viajes_stm_2024_01.csv,2024-01-29,17,1,false
2853901,6,2 HORAS,1,USUARIO CORRIENTE,1,USUARIO CORRIENTE,3459,50,C.U.T.C.S.A.,89,186,4318,2024-01-08T16:36:35.000-03:00,1,4,true,2024-01,viajes_stm_2024_01.csv,2024-01-08,16,1,false
9756259,5,1 HORA,1,USUARIO CORRIENTE,1,USUARIO CORRIENTE,1568,50,C.U.T.C.S.A.,46,128,4261,2024-01-22T06:27:50.000-03:00,1,2,true,2024-01,viajes_stm_2024_01.csv,2024-01-22,6,1,false
10016265,5,1 HORA,1,USUARIO CORRIENTE,1,USUARIO CORRIENTE,4617,20,C.O.M.E.S.A,19,538,8199,2024-01-23T07:11:03.000-03:00,1,2,true,2024-01,viajes_stm_2024_01.csv,2024-01-23,7,2,false
11365823,5,1 HORA,1,USUARIO CORRIENTE,1,USUARIO CORRIENTE,4135,10,C.O.E.T.C.,158,L16,3952,2024-01-25T15:29:00.000-03:00,3,3,true,2024-01,viajes_stm_2024_01.csv,2024-01-25,15,4,false
14063211,16,GESTION SOCIAL,4,GESTION SOCIAL,41,GESTION SOCIAL (ESC. ESP.),3091,50,C.U.T.C.S.A.,443,145,8404,2024-01-31T06:01:32.000-03:00,1,2,true,2024-01,viajes_stm_2024_01.csv,2024-01-31,6,3,false
7462413,5,1 HORA,1,USUARIO CORRIENTE,1,USUARIO CORRIENTE,2662,50,C.U.T.C.S.A.,95,199,1455,2024-01-17T10:30:55.000-03:00,2,2,true,2024-01,viajes_stm_2024_01.csv,2024-01-17,10,3,false
14597909,5,1 HORA,1,USUARIO CORRIENTE,1,USUARIO CORRIENTE,4729,50,C.U.T.C.S.A.,27,103,4145,2024-01-31T06:04:17.000-03:00,1,2,true,2024-01,viajes_stm_2024_01.csv,2024-01-31,6,3,false
11190761,5,1 HORA,1,USUARIO CORRIENTE,1,USUARIO CORRIENTE,2109,50,C.U.T.C.S.A.,76,163,4310,2024-01-25T13:12:37.000-03:00,1,2,true,2024-01,viajes_stm_2024_01.csv,2024-01-25,13,4,false


## 6. ¿Cómo identificamos una línea?

En los datos cada línea aparece con un nombre (`dsc_linea`, por ejemplo "103") y con un código (`linea_codigo`).
En gold agrupamos por **nombre**, porque es lo que la gente conoce. Las celdas de abajo revisan si eso trae problemas:

1. Si un mismo nombre tiene varios códigos o variantes (recorridos distintos de la misma línea).
2. Cuántos días funcionó cada línea en cada año. Sirve para ver líneas nuevas o que dejaron de andar.
3. Si un mismo código tiene más de un nombre.

In [0]:
from pyspark.sql import functions as F

s = spark.table(config.TABLE_SILVER).withColumn("anio", F.year("fecha"))

# 1. ¿Un mismo nombre de línea tiene varios códigos?
display(s.groupBy("dsc_linea")
         .agg(F.countDistinct("linea_codigo").alias("codigos"),
              F.countDistinct("sevar_codigo").alias("variantes"),
              F.sum("cantidad_pasajeros").alias("pasajeros"))
         .orderBy(F.desc("codigos"), F.desc("pasajeros")))

# 2. Días que funcionó cada línea en cada año (sirve para ver líneas nuevas o que se cerraron)
display(s.groupBy("dsc_linea")
         .pivot("anio", [2024, 2025, 2026])
         .agg(F.countDistinct("fecha"))
         .orderBy("dsc_linea"))

dsc_linea,codigos,variantes,pasajeros
103,1,89,27777328
183,1,17,19602341
181,1,14,19437225
G,1,17,19291713
306,1,38,19241050
145,1,40,19203807
185,1,47,19152948
405,1,15,18127674
300,1,27,17473849
163,1,20,16997778


dsc_linea,2024,2025,2026
100,364,363,302
102,364,363,302
103,366,365,304
104,364,363,302
105,366,365,304
106,362,361,301
109,364,363,302
110,364,363,302
111,364,365,304
112,364,363,302


In [0]:
# 3. ¿Un mismo código tiene más de un nombre?
display(spark.table(config.TABLE_SILVER)
        .groupBy("linea_codigo")
        .agg(F.countDistinct("dsc_linea").alias("nombres"),
             F.collect_set("dsc_linea").alias("cuales"))
        .where("nombres > 1"))

linea_codigo,nombres,cuales
